# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema}`")
spark.sql(f"USE SCHEMA `{schema}`")

In [0]:
%skip
import sys

sys.path.append("../src")
from first_bundle import taxis

taxis.find_all_taxis().show(10)

In [0]:
%skip
%sql
CREATE TABLE car_data.dev.car_data (
    car_id INT,
    make STRING,
    model STRING,
    year INT,
    color STRING,
    fuel_type STRING,
    transmission STRING,
    mileage INT,
    price DECIMAL(10,2),
    owner_count INT,
    registration_number STRING,
    created_date TIMESTAMP
);


In [0]:
%skip
%sql
INSERT INTO car_data.dev.car_data VALUES
(1, 'Toyota', 'Corolla', 2022, 'White', 'Petrol', 'Automatic', 15000, 18500.00, 1, 'GJ01AB1234', current_timestamp()),
(2, 'Honda', 'City', 2021, 'Silver', 'Petrol', 'Manual', 25000, 16000.00, 2, 'MH02CD5678', current_timestamp()),
(3, 'Hyundai', 'Creta', 2023, 'Black', 'Diesel', 'Automatic', 8000, 24000.00, 1, 'DL03EF9012', current_timestamp()),
(4, 'Maruti', 'Baleno', 2020, 'Blue', 'Petrol', 'Manual', 35000, 11000.00, 2, 'GJ05GH3456', current_timestamp()),
(5, 'Tata', 'Nexon', 2024, 'Red', 'Electric', 'Automatic', 3000, 28000.00, 1, 'KA09IJ7890', current_timestamp());


In [0]:
df = spark.sql(f"""select * from {catalog}.{schema}.car_data""")


In [0]:
from pyspark.sql.functions import count, col
analytic_df = df.groupBy("fuel_type").agg(count("*").alias("count")).orderBy(col("count").desc())

In [0]:
(
    analytic_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(f"{catalog}.{schema}.fuel_type_counts")
)